In [1]:
import os
import pandas as pd
from pathlib import Path
import gc

# Run garbage collection
gc.collect()

data_folder = Path("/kaggle/input/competitions/name-that-llm/train.csv")
data_path = os.path.join(os.getcwd(), data_folder)

print(data_path)

df = pd.read_csv(data_path)
df.head()

c:\kaggle\input\competitions\name-that-llm\train.csv


FileNotFoundError: [Errno 2] No such file or directory: 'c:\\kaggle\\input\\competitions\\name-that-llm\\train.csv'

In [ ]:
# Applying label encoder
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df["model"] = le.fit_transform(df["model"])
df.head(5)

In [ ]:
from psycopg.generators import fetch
import re
import numpy as np
import pandas as pd
import nltk
nltk.download("punkt")
from nltk.tokenize import sent_tokenize, word_tokenize

features = pd.DataFrame(index=df.index)
features["model"] = df["model"]
text_col = "text"

In [ ]:
# Character, word, and sentence counts
features["char_count"] = df[text_col].apply(len)
# Words & Sentences
def get_words_and_sents(text):
        sents = sent_tokenize(text) if len(text.strip()) > 0 else [""]
        words = word_tokenize(text) if len(text.strip()) > 0 else [""]
        return len(words), len(sents), sents, words

tokenized_info = df[text_col].apply(get_words_and_sents)
features["word_count"] = tokenized_info.apply(lambda x: x[0])
features["sent_count"] = tokenized_info.apply(lambda x: x[1])
# Sentence Length Mean & Variance
def sent_stats(info):
    _, num_sents, sents, _ = info
    if num_sents == 0:
        return 0.0, 0.0
    lengths = [len(s.split()) for s in sents]
    return float(np.mean(lengths)), float(np.var(lengths))

sent_metrics = tokenized_info.apply(sent_stats)
features["avg_sent_len"] = sent_metrics.apply(lambda x: x[0])
features["var_sent_len"] = sent_metrics.apply(lambda x: x[1])
# Avg Word Length
features["avg_word_len"] = features["char_count"] / (features["word_count"] + 1e-5)
# ----------------- 2. Markdown Stylometry -----------------
features["md_asterisk_count"] = df[text_col].apply(lambda x: x.count("*"))  # Bold or italics (* or **)
# ----------------- 3. Punctuation Fingerprint -----------------
features["punct_colon"] = df[text_col].apply(lambda x: x.count(":"))
features["punct_semicolon"] = df[text_col].apply(lambda x: x.count(";"))
features["punct_em_dash"] = df[text_col].apply(lambda x: x.count("—") + x.count("--"))
features["punct_exclamation"] = df[text_col].apply(lambda x: x.count("!"))
features["punct_question"] = df[text_col].apply(lambda x: x.count("?"))
features["punct_comma"] = df[text_col].apply(lambda x: x.count(","))
features["punct_period"] = df[text_col].apply(lambda x: x.count("."))
features["punct_quotes"] = df[text_col].apply(lambda x: x.count('"') + x.count("'"))
features["punct_parens"] = df[text_col].apply(lambda x: x.count("(") + x.count(")"))
features["punct_brackets"] = df[text_col].apply(lambda x: x.count("[") + x.count("]"))
# Punctuation Density (normalized by word count)
total_puncts = (features["punct_colon"] + features["punct_semicolon"] + 
                features["punct_comma"] + features["punct_period"] + features["punct_exclamation"])
features["punct_density"] = total_puncts / (features["word_count"] + 1e-5)
# ----------------- 4. Capitalization & Casing -----------------
features["upper_char_count"] = df[text_col].apply(lambda x: sum(1 for c in x if c.isupper()))
features["upper_char_ratio"] = features["upper_char_count"] / (features["char_count"] + 1e-5)
features["all_caps_words"] = df[text_col].apply(lambda x: len(re.findall(r"\b[A-Z]{2,}\b", x)))
features["camel_case_count"] = df[text_col].apply(lambda x: len(re.findall(r"\b[a-z]+[A-Z][a-zA-Z]*\b", x)))
# ----------------- 5. Lexical Diversity (Type-Token Ratio) -----------------
def type_token_ratio(info):
    num_words, _, _, words = info
    if num_words == 0:
        return 0.0
    unique_words = set(w.lower() for w in words if w.isalnum())
    return len(unique_words) / (num_words + 1e-5)

features["lexical_diversity_ttr"] = tokenized_info.apply(type_token_ratio)
# Digits / Numbers presence
features["digit_count"] = df[text_col].apply(lambda x: sum(1 for c in x if c.isdigit()))
features["digit_ratio"] = features["digit_count"] / (features["char_count"] + 1e-5)
features.head()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

#  Train-Test Split (stratified)
df_train, df_test = train_test_split(features, test_size=0.2, random_state=42, stratify=features["model"])
y_train = df_train["model"].values
y_test = df_test["model"].values
X_train_pre = df_train.iloc[:, 1:]
X_test_pre = df_test.iloc[:, 1:]

X_train_pre.shape, X_test_pre.shape
# X_train.head()
encoder = StandardScaler()
X_train = encoder.fit_transform(X_train_pre)
X_test = encoder.transform(X_test_pre)
X_train.shape, X_test.shape
X_train

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score, confusion_matrix
clf = LogisticRegression(max_iter=1000, C=2.0, verbose=1)

clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}")
print(f"Recall Score: {recall_score(y_test, y_pred, average='macro'):.4f}")
# print(f"Confusion Matrix:\n{confusion_matrix(y_test, y_pred)}")



In [ ]:
test = pd.read_csv(r"/kaggle/input/competitions/name-that-llm/test.csv")
test.head()

features = pd.DataFrame(index=df.index)
# Character, word, and sentence counts
features["char_count"] = test[text_col].apply(len)
# Words & Sentences
def get_words_and_sents(text):
        sents = sent_tokenize(text) if len(text.strip()) > 0 else [""]
        words = word_tokenize(text) if len(text.strip()) > 0 else [""]
        return len(words), len(sents), sents, words

tokenized_info = test[text_col].apply(get_words_and_sents)
features["word_count"] = tokenized_info.apply(lambda x: x[0])
features["sent_count"] = tokenized_info.apply(lambda x: x[1])
# Sentence Length Mean & Variance
def sent_stats(info):
    _, num_sents, sents, _ = info
    if num_sents == 0:
        return 0.0, 0.0
    lengths = [len(s.split()) for s in sents]
    return float(np.mean(lengths)), float(np.var(lengths))

sent_metrics = tokenized_info.apply(sent_stats)
features["avg_sent_len"] = sent_metrics.apply(lambda x: x[0])
features["var_sent_len"] = sent_metrics.apply(lambda x: x[1])
# Avg Word Length
features["avg_word_len"] = features["char_count"] / (features["word_count"] + 1e-5)
# ----------------- 2. Markdown Stylometry -----------------
features["md_asterisk_count"] = df[text_col].apply(lambda x: x.count("*"))  # Bold or italics (* or **)
# ----------------- 3. Punctuation Fingerprint -----------------
features["punct_colon"] = df[text_col].apply(lambda x: x.count(":"))
features["punct_semicolon"] = df[text_col].apply(lambda x: x.count(";"))
features["punct_em_dash"] = df[text_col].apply(lambda x: x.count("—") + x.count("--"))
features["punct_exclamation"] = df[text_col].apply(lambda x: x.count("!"))
features["punct_question"] = df[text_col].apply(lambda x: x.count("?"))
features["punct_comma"] = df[text_col].apply(lambda x: x.count(","))
features["punct_period"] = df[text_col].apply(lambda x: x.count("."))
features["punct_quotes"] = df[text_col].apply(lambda x: x.count('"') + x.count("'"))
features["punct_parens"] = df[text_col].apply(lambda x: x.count("(") + x.count(")"))
features["punct_brackets"] = df[text_col].apply(lambda x: x.count("[") + x.count("]"))
# Punctuation Density (normalized by word count)
total_puncts = (features["punct_colon"] + features["punct_semicolon"] + 
                features["punct_comma"] + features["punct_period"] + features["punct_exclamation"])
features["punct_density"] = total_puncts / (features["word_count"] + 1e-5)
# ----------------- 4. Capitalization & Casing -----------------
features["upper_char_count"] = df[text_col].apply(lambda x: sum(1 for c in x if c.isupper()))
features["upper_char_ratio"] = features["upper_char_count"] / (features["char_count"] + 1e-5)
features["all_caps_words"] = df[text_col].apply(lambda x: len(re.findall(r"\b[A-Z]{2,}\b", x)))
features["camel_case_count"] = df[text_col].apply(lambda x: len(re.findall(r"\b[a-z]+[A-Z][a-zA-Z]*\b", x)))
# ----------------- 5. Lexical Diversity (Type-Token Ratio) -----------------
def type_token_ratio(info):
    num_words, _, _, words = info
    if num_words == 0:
        return 0.0
    unique_words = set(w.lower() for w in words if w.isalnum())
    return len(unique_words) / (num_words + 1e-5)

features["lexical_diversity_ttr"] = tokenized_info.apply(type_token_ratio)
# Digits / Numbers presence
features["digit_count"] = df[text_col].apply(lambda x: sum(1 for c in x if c.isdigit()))
features["digit_ratio"] = features["digit_count"] / (features["char_count"] + 1e-5)
features.shape

In [ ]:
X_test_t = features.iloc[:, :].copy()
X_test = encoder.transform(X_test_t)
X_test_scaled_df = pd.DataFrame(X_test, columns=X_test_t.columns, index=X_test_t.index)
X_test_scaled_df

In [ ]:

predictions = clf.predict(X_test)
print(predictions)
print(le.inverse_transform(predictions)) 
predictions_f = le.inverse_transform(predictions)
# Submission file 
submission = pd.DataFrame({
    "id": test["id"],
    "model": predictions_f
})

submission.to_parquet("submission.parquet", index=False)
print(f"Saved submission.parquet with {len(submission)} rows.")
submission.head(10)